# 06 — WebGPU acceleration

*Phase 3 of the [revival roadmap](../docs/impl-plans/2026-05-11_phase-3-webgpu.md): the third concrete `KernelBackend` adapter, lifting named-axis tensor algebra onto the GPU through [WebGPU](https://www.w3.org/TR/webgpu/) without any proprietary toolchain.*

**This notebook is a narration of shipped code, executed in CI without a GPU.** Phase 3 dispatch wiring landed on the maintainer's RTX 3090 in [PRs #60 / #61 / #62](https://github.com/uyuutosa/tensor/pulls?q=is%3Apr+is%3Amerged) — 12 of the 15 `KernelBackend` methods now dispatch real Dawn compute on `float`. The notebook stays GPU-free because [notebook CI](https://github.com/uyuutosa/tensor/blob/develop/.github/workflows/notebook-ci.yml) runs on stock GitHub-hosted runners (no GPU); the reader executes the WGSL-printing cells and reads the dispatch-design markdown. To run the kernels live, build with `-DTENSOR_KERNEL_BACKEND=webgpu` after `vcpkg install 'dawn[core,vulkan]'` (per [`README.md`](../README.md) §Quickstart) and run the [test suite](../tests/test_webgpu_backend.cpp).

## What you will get from this notebook

1. You see the **nine committed WGSL kernels** (four element-wise binary + four unary + one tiled GEMM + a templated broadcast body specialised for three operators) and understand what each does.
2. You understand the **dispatch wiring design** — how `tensor::core::backend::webgpu::Backend::add(a, b)` turns into a Dawn `webgpu_cpp.h` (`wgpu::Device::CreateBuffer` / `CreateComputePipeline` / `Queue::Submit`) sequence (per [ADR-0016](../docs/arc42/09-decisions/0016-substrate-refinement-drop-gpu-cpp-talk-to-dawn-directly.md)).
3. You see the **substrate trade-offs** captured by [ADR-0014](../docs/arc42/09-decisions/0014-external-substrate-strategy.md) and refined by [ADR-0016](../docs/arc42/09-decisions/0016-substrate-refinement-drop-gpu-cpp-talk-to-dawn-directly.md): Dawn via vcpkg; gpu.cpp tried and dropped after a 14-month ABI drift; xeus-cpp replaced xeus-cling.
4. You see the **honest performance picture** as measured on RTX 3090 ([2026-05-12 backend perf comparison](../docs/reports/2026-05-11_backend-performance-comparison.md)): WebGPU **149× over reference / 3.6× over Eigen on matmul 512³**, but **slower than CPU on add 1M elements** because host↔device transfer dominates.

## Prerequisites

**[`00_intro.ipynb`](./00_intro.ipynb) — named-axis fundamentals.** No autograd knowledge required: this notebook covers *forward* execution on GPU only. The autograd backward pass is in [`05_autograd-from-scratch.ipynb`](./05_autograd-from-scratch.ipynb) and runs independently of the kernel backend — `Variable<T>` does not care whether the underlying ops dispatched on reference, Eigen, or WebGPU.

If you arrived here from a "WebGPU C++" search and want to see the WGSL kernels + dispatch design without learning autograd first, that is supported. If you arrived from a "named-tensor autograd" search and want autograd on the GPU, you also need notebook 05 (the same `Variable<T>` machinery composes with any `KernelBackend` adapter).

Sibling notebook 08 ([`08_swappable-backends.ipynb`](./08_swappable-backends.ipynb)) demonstrates the same Hexagonal-lite payoff for the reference + Eigen pair — it is the closest analogue to the architectural argument this notebook makes for WebGPU.

## 1. The Phase 3 sub-structure

Phase 3 (WebGPU adapter) ships in six milestones; all six are now landed.

| Milestone | Content | PR | Status |
| --------- | ------- | -- | ------ |
| P3.M1 | [ADR-0012](../docs/arc42/09-decisions/0012-webgpu-adapter-implementation-design.md) — WebGPU adapter implementation design | [#32](https://github.com/uyuutosa/tensor/pull/32) | shipped |
| P3.M2 | `webgpu::Backend` stub + CMake plumbing (delegates every method to `reference::Backend`) | [#38](https://github.com/uyuutosa/tensor/pull/38) | shipped |
| **P3.M3.1** | **WGSL sources for 4 binary element-wise kernels** (`add`/`sub`/`mul`/`div`) | [#43](https://github.com/uyuutosa/tensor/pull/43) | shipped |
| **P3.M3.3** | **WGSL sources for 4 unary element-wise kernels** (`exp`/`log`/`relu`/`neg`) | [#44](https://github.com/uyuutosa/tensor/pull/44) | shipped |
| **P3.M4.1** | **WGSL source for tiled GEMM kernel** (covers matvec + matmul) | [#46](https://github.com/uyuutosa/tensor/pull/46) | shipped |
| **P3.M3.2** | **Dawn dispatch wiring for element-wise (replaces stub delegation)** | [#60](https://github.com/uyuutosa/tensor/pull/60) | shipped |
| **P3.M4.2** | **Dawn dispatch wiring for GEMM** | [#61](https://github.com/uyuutosa/tensor/pull/61) | shipped |
| **P3.M5** | **Broadcast WGSL + dispatch wiring (3 ops)** | [#62](https://github.com/uyuutosa/tensor/pull/62) | shipped |
| P3.M6 | This notebook (`06_webgpu-acceleration.ipynb`) — closes Phase 3 | [#56](https://github.com/uyuutosa/tensor/pull/56) | shipped |

The split between `*.1` / `*.3` (committed WGSL source) and `*.2` (Dawn dispatch) is per the [Phase 3 impl-plan addendum](../docs/impl-plans/2026-05-11_phase-3-webgpu.md). Both detailed-design docs ([element-wise](../docs/detailed-design/webgpu-element-wise-kernels.md) and [GEMM](../docs/detailed-design/webgpu-gemm-kernel.md)) plus the third ([broadcast](../docs/detailed-design/webgpu-broadcast-kernels.md)) document the dispatch sequence at line-resolved detail. Of the 15-method [`KernelBackend`](../docs/detailed-design/kernel-backend-port.md) port, 12 methods dispatch real GPU compute on `float`; the remaining 3 (`reduce_sum`, `unbroadcast`, non-simple-GEMM `contract`) delegate to reference, matching the Eigen adapter's scope.

## 2. The nine committed WGSL kernels

All sources live under [`include/tensor/core/backend/webgpu_wgsl.hpp`](../include/tensor/core/backend/webgpu_wgsl.hpp) as `constexpr std::string_view` constants. Each is templated on `{{precision}}` and `{{workgroupSize}}` placeholders the project substitutes itself in [`webgpu_detail/dispatch.hpp::substitute_wgsl`](../include/tensor/core/backend/webgpu_detail/dispatch.hpp) (the convention originated with gpu.cpp's `KernelCode`; per [ADR-0016](../docs/arc42/09-decisions/0016-substrate-refinement-drop-gpu-cpp-talk-to-dawn-directly.md) the project now does the substitution directly rather than going through gpu.cpp's wrapper).

In [ ]:
#pragma cling add_include_path("../include")
#include <iostream>
#include <tensor/core/backend/webgpu_wgsl.hpp>

namespace wgsl = tensor::core::backend::webgpu::wgsl;

std::cout << "==== kAddF32 ====" << wgsl::kAddF32 << "\n";

### Element-wise binary structure

```
@group(0) @binding(0) var<storage, read>       a   : array<{{precision}}>;
@group(0) @binding(1) var<storage, read>       b   : array<{{precision}}>;
@group(0) @binding(2) var<storage, read_write> out : array<{{precision}}>;

@compute @workgroup_size({{workgroupSize}})
fn main(@builtin(global_invocation_id) gid: vec3<u32>) {
    let i = gid.x;
    if (i >= arrayLength(&out)) { return; }
    out[i] = a[i] <op> b[i];
}
```

The four binary kernels (`kAddF32`, `kSubF32`, `kMulF32`, `kDivF32`) differ only in the `<op>` — `+`, `-`, `*`, `/`. They share:

- Three storage bindings: read-only `a`, read-only `b`, read-write `out`.
- A 1-D workgroup with `{{workgroupSize}}` = 256 threads (matches Dawn's canonical choice; substituted by `gpu::KernelCode`).
- `arrayLength(&out)` for the bound check — works because `out`'s size at dispatch time is what the host code uploaded.

### Element-wise unary structure

The four unary kernels (`kExpF32`, `kLogF32`, `kReluF32`, `kNegF32`) are identical except for **two storage bindings** (input + output) and the body. Notably, `kReluF32` uses `max(a[i], 0.0)` rather than a branch — `max` maps to a single GPU instruction on every Dawn backend (Vulkan / Metal / D3D12).

### The tiled GEMM kernel

`kGemmF32` is one readable kernel that covers both matvec (rank-2 × rank-1) and matmul (rank-2 × rank-2) with one shared axis. The full design is in [`webgpu-gemm-kernel.md`](../docs/detailed-design/webgpu-gemm-kernel.md); the headline structure:

```
struct Params { M : u32, N : u32, K : u32 };
// bindings: a (M×K), b (K×N), out (M×N), p (uniform Params)

const TILE_M : u32 = 16u;
const TILE_N : u32 = 16u;
const TILE_K : u32 = 16u;

var<workgroup> shA : array<array<{{precision}}, TILE_K>, TILE_M>;
var<workgroup> shB : array<array<{{precision}}, TILE_N>, TILE_K>;

@compute @workgroup_size(TILE_N, TILE_M, 1)
fn main(...) {
    var acc : {{precision}} = 0.0;
    for (var t : u32 = 0u; t < nTiles; t = t + 1u) {
        // 1. cooperative load: 16×16 tile of A + 16×16 tile of B into shared memory
        workgroupBarrier();
        for (var k : u32 = 0u; k < TILE_K; k = k + 1u) {
            acc = acc + shA[lrow][k] * shB[k][lcol];
        }
        workgroupBarrier();
    }
    if (row < p.M && col < p.N) { out[row * p.N + col] = acc; }
}
```

Why one kernel for matvec and matmul? The canonical-reference-quality framing ([ADR-0015](../docs/arc42/09-decisions/0015-aspire-to-canonical-reference-quality-not-self-anoint.md), superseding ADR-0013) prefers one readable kernel over two specialised ones. A dedicated matvec kernel would extract more throughput when N = 1, but at the cost of doubling the surface area to read, test, and re-derive. The matvec inefficiency (15 / 16 threads idle in a 16-wide workgroup when N = 1) is documented in the design doc's §4 rather than engineered away. §5's RTX 3090 measurements confirm this trade-off — matvec 1024² loses to Eigen by a wide margin, while matmul 512³ wins by 3.6× over Eigen.

In [ ]:
// Confirm the tile constants the kernel source hard-codes match the
// C++ side's expectations — text-validated by tests/test_webgpu_wgsl.cpp.
std::cout << "kGemmTileM = " << wgsl::kGemmTileM << "\n";
std::cout << "kGemmTileN = " << wgsl::kGemmTileN << "\n";
std::cout << "kGemmTileK = " << wgsl::kGemmTileK << "\n";
std::cout << "kDefaultWorkgroupSize = " << wgsl::kDefaultWorkgroupSize << "\n";

## 3. The dispatch wiring (what P3.M3.2 / P3.M4.2 / P3.M5 landed)

Prior to PR #60, every method on `webgpu::Backend` delegated to a private `reference::Backend ref_` member. PRs #60 (element-wise) / #61 (GEMM) / #62 (broadcast) replaced each `float` delegation with a Dawn dispatch using Dawn's own `<webgpu/webgpu_cpp.h>` ([ADR-0016](../docs/arc42/09-decisions/0016-substrate-refinement-drop-gpu-cpp-talk-to-dawn-directly.md)). The full design with line-resolved references is in the three detailed-design docs ([element-wise](../docs/detailed-design/webgpu-element-wise-kernels.md), [GEMM](../docs/detailed-design/webgpu-gemm-kernel.md), [broadcast](../docs/detailed-design/webgpu-broadcast-kernels.md)).

### Element-wise binary dispatch sequence (e.g. `add`)

```cpp
template <class T>
DynamicTensor<T> Backend::add(DynamicTensor<T> const& a,
                              DynamicTensor<T> const& b) const {
    if constexpr (!std::is_same_v<T, float>) {
        return ref_.add(a, b);  // f32-only MVP; delegate non-float
    } else {
        auto& ctx = WebGPUContext::current();          // thread_local singleton
        auto const n_bytes = a.total() * sizeof(T);

        // 3 storage buffers: a (input), b (input), out (storage, copy_src)
        wgpu::Buffer gA   = create_storage_buffer(ctx.device, n_bytes, a.data());
        wgpu::Buffer gB   = create_storage_buffer(ctx.device, n_bytes, b.data());
        wgpu::Buffer gOut = create_storage_buffer(ctx.device, n_bytes, nullptr);

        // Substitute {{precision}} → "f32", {{workgroupSize}} → "256"
        std::string source = substitute_wgsl(wgsl::kAddF32, "f32",
                                             wgsl::kDefaultWorkgroupSize);

        wgpu::ShaderModule module = create_shader_module(ctx.device, source);
        wgpu::ComputePipeline pipeline =
            create_compute_pipeline(ctx.device, module, "main");

        wgpu::BindGroup bind = create_bind_group(ctx.device,
            pipeline.GetBindGroupLayout(0), {{gA, 0}, {gB, 0}, {gOut, 0}});

        // Submit + wait
        wgpu::CommandEncoder enc = ctx.device.CreateCommandEncoder({});
        {
            auto pass = enc.BeginComputePass();
            pass.SetPipeline(pipeline);
            pass.SetBindGroup(0, bind);
            uint32_t workgroups = (n + 255) / 256;     // ceil-div by workgroupSize
            pass.DispatchWorkgroups(workgroups, 1, 1);
            pass.End();
        }
        wgpu::CommandBuffer cmd = enc.Finish();
        ctx.queue.Submit(1, &cmd);

        DynamicTensor<T> out{a.shape()};
        copy_buffer_to_host(ctx, gOut, out.data(), n_bytes);
        return out;
    }
}
```

Substitute `kAddF32` → `kSubF32` / `kMulF32` / `kDivF32` for the other three binary operators; for the four unary operators drop the `gB` upload and use a two-binding layout. Identical structural shape.

### GEMM dispatch sequence

Adds a `Params { M, N, K }` uniform buffer and a 2-D `DispatchWorkgroups(ceil(N/16), ceil(M/16), 1)`. Same Buffer / ShaderModule / Pipeline / BindGroup / Submit / wait pattern. Full code in [`webgpu-gemm-kernel.md §3`](../docs/detailed-design/webgpu-gemm-kernel.md).

### Broadcast dispatch sequence (the storage-vs-uniform discovery)

Same shape, but `BroadcastParams` (`rank`, `extents[8]`, `srcA_axes[8]`, `srcB_axes[8]` with `0xFFFFFFFF` as `npos`) is bound as a **storage** buffer rather than a uniform — `array<u32, 8>` in a uniform buffer has 16-byte stride per element (std140-like), which mismatches C++ packed layout. The `webgpu-broadcast-kernels.md §4.3.1` writeup records the bug + fix.

## 4. Substrate choices (why these and not others)

[ADR-0014](../docs/arc42/09-decisions/0014-external-substrate-strategy.md) bundles four substrate decisions; [ADR-0016](../docs/arc42/09-decisions/0016-substrate-refinement-drop-gpu-cpp-talk-to-dawn-directly.md) refined point 2 after the [2026-05-12 gpu.cpp / Dawn ABI drift report](../docs/reports/2026-05-12_gpu-cpp-dawn-abi-drift-and-raw-dawn-smoke.md).

1. **Dawn directly via `webgpu_cpp.h`** ([ADR-0016](../docs/arc42/09-decisions/0016-substrate-refinement-drop-gpu-cpp-talk-to-dawn-directly.md)). The project initially chose to vendor [AnswerDotAI/gpu.cpp](https://github.com/AnswerDotAI/gpu.cpp) as a thin wrapper (per ADR-0014 §Decision Outcome point 2). The first real GPU smoke against the vcpkg Dawn port (`20260410.140140`, 2026-04) revealed that gpu.cpp@0.2.0 is **14 months behind Dawn's async-callback API migration** (six call sites in `gpu.hpp` require rewrites). Rather than fork-and-patch gpu.cpp, ADR-0016 dropped it: the project now talks to Dawn through Dawn's own `<webgpu/webgpu_cpp.h>` RAII wrapper (Google-maintained, always synchronised with the linked binary). The vendored `third_party/gpu_cpp/` was removed in [PR #65](https://github.com/uyuutosa/tensor/pull/65). The vendoring discipline is *vindicated*, not abandoned: it caught the ABI drift at our build, not in a user's hand.

2. **Dawn comes from vcpkg.** Port `20260410.140140` (refreshed 2026-04-20). The maintainer's verification path is local `vcpkg install 'dawn[core,vulkan]'` (the `x11` / `gl` / `gles` default features are excluded to skip the `libx11-xcb-dev` dependency). `find_package(Dawn CONFIG REQUIRED)` + `target_link_libraries(... dawn::webgpu_dawn)` is gated on `TENSOR_KERNEL_BACKEND=webgpu` in [`CMakeLists.txt`](../CMakeLists.txt). A future vcpkg baseline bump remains a tracked option for users who want Dawn without the separate install step; the `webgpu` manifest feature in [`vcpkg.json`](../vcpkg.json) declares the dependency shape.

3. **No `wgpu-native`.** It has no vcpkg port and would require a Rust toolchain in CI. ADR-0014 keeps it as a documented opt-in fallback for environments where Dawn is fussy (some Linux mesa stacks).

4. **xeus-cpp 0.10+ is the notebook kernel** (replacing the frozen xeus-cling). Per [ADR-0014 §3](../docs/arc42/09-decisions/0014-external-substrate-strategy.md). The kernel name (`xcpp20`) is the same, so this notebook works on either kernel; xeus-cling is kept as a `legacy-xeus-cling` smoke job in [notebook CI](../.github/workflows/notebook-ci.yml) that runs only `00_intro.ipynb`.

## 5. Performance measurements (RTX 3090, 2026-05-12)

Per [ADR-0001](../docs/arc42/09-decisions/0001-pivot-to-educational-named-axis-dsl.md) and the [canonical-reference-quality framing of ADR-0015](../docs/arc42/09-decisions/0015-aspire-to-canonical-reference-quality-not-self-anoint.md), performance is a quality-4 concern that **must not block** quality 1–3 (clarity, correctness, portability). The WebGPU adapter exists primarily as the architectural payoff of the Hexagonal-lite design — it is *possible* without changing the Domain. That said, the architectural payoff comes with real numbers.

Measured on **NVIDIA RTX 3090, driver 560.28.03, Dawn 2026-04, g++ 12.3 -O3** (per [bench/bench.cpp](../bench/bench.cpp), recorded in [`docs/reports/2026-05-11_backend-performance-comparison.md`](../docs/reports/2026-05-11_backend-performance-comparison.md) 2026-05-12 subsection):

| Operation | Reference | Eigen | WebGPU | Verdict |
| --------- | --------- | ----- | ------ | ------- |
| **matmul 512³** | 572 ms | **13.6 ms** (42× ref) | **3.8 ms** (149× ref, 3.6× eigen) | Compute amortises transfer; WebGPU wins decisively. |
| matvec 1024² | 3.1 ms | 0.15 ms (21×) | 6.4 ms | Compute ≈ transfer; 1-D effective dispatch wastes 15/16 threads — documented matvec inefficiency. WebGPU loses. |
| add 1M elements | 1.1 ms | 0.72 ms (1.5×) | 7.0 ms | Host↔device transfer of ~12 MB roundtrip dominates compute (<0.1 ms). WebGPU loses by a wide margin. |

**Conclusion**: WebGPU is competitive with optimised CPU BLAS for compute-bound ops at moderate-to-large sizes. For small element-wise ops, CPU wins on round-trip alone. The canonical learner-facing takeaway, from tutorial 06's framing, is *not* that WebGPU is universally faster — it is that **the same Domain code reaches the GPU when the workload warrants it**, with one CMake variable's worth of opt-in cost.

## 6. What's needed to bring live execution into CI

The dispatch wiring **already runs on the maintainer's RTX 3090** ([PRs #60 / #61 / #62](https://github.com/uyuutosa/tensor/pulls?q=is%3Apr+is%3Amerged), [test_webgpu_backend.cpp](../tests/test_webgpu_backend.cpp) cross-validates 154 / 154 cases / 4852 / 4852 assertions vs reference within `1e-5` / `1e-3` tolerances). The notebook stays GPU-free because the *notebook CI* runs on stock GitHub-hosted runners.

To bring live execution into the bundled CI:

1. **A self-hosted GitHub Actions runner with a Dawn-compatible GPU.** Per [ADR-0012 §Decision Outcome point 6](../docs/arc42/09-decisions/0012-webgpu-adapter-implementation-design.md), the project's CI policy is compile-only on generic runners + numerical-agreement on a self-hosted runner. The maintainer's RTX 3090 is the obvious candidate but adding it to GHA is a separate (maintainer-only) configuration step.

2. **Optional: a vcpkg baseline bump that includes `dawn[core,vulkan]`** so the runner can `vcpkg install --x-feature=webgpu` without a separate `vcpkg install 'dawn[core,vulkan]'` step. The current `webgpu` manifest feature in [`vcpkg.json`](../vcpkg.json) declares the dependency shape but the default `builtin-baseline` predates Dawn — a baseline bump (or a `vcpkg-overlay-ports` override) closes the gap.

Until those land, the WebGPU adapter's correctness is verified locally by the maintainer before each release per the [Phase 4 release rehearsal](../docs/reports/2026-05-11_phase-4-release-rehearsal.md) §3 checklist. The Risk §11 line for "no real-GPU verification" (R-Q2) [demoted from 🔴 to 🟢 in PR #76](https://github.com/uyuutosa/tensor/pull/76) accordingly.

## 7. Cross-references

**Detailed designs (line-resolved):**
- [`docs/detailed-design/webgpu-element-wise-kernels.md`](../docs/detailed-design/webgpu-element-wise-kernels.md) — P3.M3 element-wise design + dispatch.
- [`docs/detailed-design/webgpu-gemm-kernel.md`](../docs/detailed-design/webgpu-gemm-kernel.md) — P3.M4 GEMM design + dispatch.
- [`docs/detailed-design/webgpu-broadcast-kernels.md`](../docs/detailed-design/webgpu-broadcast-kernels.md) — P3.M5 broadcast design + dispatch (includes the storage-vs-uniform / `DynamicShape::size()` bug discovery).
- [`docs/detailed-design/kernel-backend-port.md`](../docs/detailed-design/kernel-backend-port.md) — the 15-method `KernelBackend` port that decouples the Domain from execution; lists per-adapter behaviour for each method.

**ADRs anchoring this notebook:**
- [ADR-0006](../docs/arc42/09-decisions/0006-adopt-webgpu-as-gpu-backend.md) — WebGPU as the GPU backend choice.
- [ADR-0011](../docs/arc42/09-decisions/0011-kernel-backend-port-api.md) — `KernelBackend` port API.
- [ADR-0012](../docs/arc42/09-decisions/0012-webgpu-adapter-implementation-design.md) — WebGPU adapter implementation design.
- [ADR-0014](../docs/arc42/09-decisions/0014-external-substrate-strategy.md) — external substrate strategy.
- [ADR-0016](../docs/arc42/09-decisions/0016-substrate-refinement-drop-gpu-cpp-talk-to-dawn-directly.md) — substrate refinement: drop gpu.cpp; talk to Dawn directly via `webgpu_cpp.h`.

**Reports & plans:**
- [Phase 3 impl-plan](../docs/impl-plans/2026-05-11_phase-3-webgpu.md) — full milestone list.
- [External substrate research (2026-05-11)](../docs/reports/2026-05-11_external-substrate-research.md) — why Dawn / xeus-cpp / stdBLAS.
- [gpu.cpp / Dawn ABI drift report (2026-05-12)](../docs/reports/2026-05-12_gpu-cpp-dawn-abi-drift-and-raw-dawn-smoke.md) — the discovery that triggered ADR-0016.
- [Backend perf comparison report (2026-05-11, refreshed 2026-05-12)](../docs/reports/2026-05-11_backend-performance-comparison.md) — the source for §5's RTX 3090 numbers.
- [Phase 4 release rehearsal (2026-05-11)](../docs/reports/2026-05-11_phase-4-release-rehearsal.md) — recommends Option 3 narration mode for tutorial 06 (i.e. this notebook).

**Shipped source:**
- [`include/tensor/core/backend/webgpu_wgsl.hpp`](../include/tensor/core/backend/webgpu_wgsl.hpp) — the nine kernel sources walked through above.
- [`include/tensor/core/backend/webgpu.hpp`](../include/tensor/core/backend/webgpu.hpp) — the `Backend` adapter (12 of 15 methods dispatch real Dawn compute on `float`; the rest delegate to `reference`).
- [`include/tensor/core/backend/webgpu_detail/context.hpp`](../include/tensor/core/backend/webgpu_detail/context.hpp) — `WebGPUContext` thread-local singleton managing Dawn `Instance` / `Adapter` / `Device` / `Queue` lifetime.
- [`include/tensor/core/backend/webgpu_detail/dispatch.hpp`](../include/tensor/core/backend/webgpu_detail/dispatch.hpp) — `dispatch_element_wise<N>`, `dispatch_gemm`, `dispatch_broadcast`, `substitute_wgsl`, `copy_buffer_to_host`.

**Sibling notebooks:**
- [`tutorials/00_intro.ipynb`](./00_intro.ipynb) — named-axis fundamentals.
- [`tutorials/01_formula-is-the-program.ipynb`](./01_formula-is-the-program.ipynb) — `_tex` UDL end-to-end (orthogonal to the kernel backend).
- [`tutorials/05_autograd-from-scratch.ipynb`](./05_autograd-from-scratch.ipynb) — autograd primitive-by-primitive (composes with any backend).
- [`tutorials/07_mlp-on-toy.ipynb`](./07_mlp-on-toy.ipynb) — capstone training loop.
- [`tutorials/08_swappable-backends.ipynb`](./08_swappable-backends.ipynb) — Hexagonal-lite payoff with reference + Eigen.